# Football

In [ ]:
#| include: false

import warnings
warnings.filterwarnings('ignore')

"I keep telling people, the eye in the sky doesn't lie." -- Howard Cosell

"Your best pass play is a run, and your best run play is a pass." -- Darrell Royal, former head coach at the University of Texas

"If you want to win, do ordinary things better than anyone else does them day in and day out." -- Chuck Noll, four-time Super Bowl winning head coach of the Pittsburgh Steelers

## Python Modules

We will use the following Python modules in this chapter that we have used in previous chapters:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

We will also use the following modules in this chapter:

In [ ]:
import nfl_data_py as nfl
from pygam import LogisticGAM, s
from scipy import stats

If you do not remember how to install new modules in Python, refer to Chapter 2 where there is a detailed discussion. The `nfl_data_py` module provides access to NFL play-by-play data, schedules, rosters, and team information. The `pygam` module provides generalized additive models (GAMs), which we will use to model nonlinear relationships in field goal data.

In [ ]:
#| echo: false
#| eval: true

sns.set_theme(style="whitegrid")

## If You Don't Know Anything About Football

American football is played between two teams of eleven players each on a rectangular field 100 yards long with 10-yard end zones at each end. The objective of the game is to score more points than the opposing team. A team can score by advancing the ball into the opponent's end zone for a **touchdown** (6 points, plus an extra point attempt), kicking the ball through the opponent's goalposts for a **field goal** (3 points), or tackling an opposing ball carrier in their own end zone for a **safety** (2 points).

The team with possession of the ball (the **offense**) has four attempts, called **downs**, to advance the ball at least 10 yards. If the offense successfully moves the ball 10 or more yards, they earn a new set of four downs. If the offense fails to advance the necessary yardage in four downs, possession is turned over to the other team. Because of this, teams typically **punt** (kick the ball away) on fourth down to push the opposing team further from their end zone, or attempt a field goal if they are close enough. However, as we will see in this chapter, analytics has begun to challenge this conventional wisdom.

Each game consists of four 15-minute quarters, with a halftime intermission between the second and third quarters. The game clock stops for incomplete passes, when a player goes out of bounds, and for various other reasons. An NFL regular season consists of 18 weeks, with each team playing 17 games and having one bye week. The season is followed by a single-elimination playoff tournament culminating in the Super Bowl.

![A standard American football field. The field is 100 yards long with 10-yard end zones at each end.](fig/nfl-field.png){#fig-nfl-field}

| Statistic | Description |
|:---|:---|
| Yards | Total distance gained by the offense |
| Touchdowns (TD) | Scoring plays worth 6 points |
| Field Goals (FG) | Kicks through the goalposts worth 3 points |
| First Downs | Successful conversions of 10+ yards for a new set of downs |
| Turnovers | Interceptions or fumbles that result in a change of possession |
| Sacks | Tackles of the quarterback behind the line of scrimmage |
| Passer Rating | A composite metric evaluating quarterback passing performance |
| Yards per Play | Average yards gained per offensive play |
| Time of Possession | Total time a team controls the ball |
| Third Down Conversion Rate | Percentage of third downs successfully converted |

: Basic Football Metrics. {#tbl-football-metrics}

## Why Football Analytics?

Football analytics has a shorter history than its counterpart in baseball, but it has grown rapidly in the 21st century. While baseball's analytics revolution was catalyzed by Bill James and the sabermetrics movement beginning in the 1970s and 1980s, football's analytical awakening came later, driven largely by the availability of play-by-play data and advances in tracking technology.

One of the earliest and most influential contributions to football analytics was the work of David Romer, an economist at UC Berkeley, who published a paper in 2006 titled *Do Firms Maximize? Evidence from Professional Football*. Romer analyzed fourth-down decision-making in the NFL and found that teams were systematically too conservative --- punting or kicking field goals in situations where going for it on fourth down would have been the better strategy in terms of expected points. His work highlighted the gap between what coaches were doing and what the data suggested they should do.

Since then, football analytics has expanded into numerous areas. Teams now employ data scientists and analysts who work on everything from play-calling optimization and player evaluation to injury prediction and draft strategy. The public availability of play-by-play data through projects like `nflfastR` (in R) and `nfl_data_py` (in Python) has democratized access to NFL data and enabled a thriving community of independent researchers and analysts.

Three areas where analytics has had a particularly significant impact on football strategy are **fourth down decision-making**, **expected points models**, and **win probability models**. We will explore all three of these topics in this chapter.

## The Coach's Dilemma: What to Do on Fourth Down?

One of the most consequential decisions in football occurs on fourth down. Traditionally, coaches have followed a simple heuristic: if you are close enough to the opponent's end zone, attempt a field goal; if you are too far away, punt; and only go for it (attempt to gain a first down) in desperate situations late in the game. This conventional approach has been ingrained in football culture for decades, and coaches who deviate from it often face intense scrutiny from fans and media --- regardless of the outcome.

However, analytics has shown that this conservative approach often leaves points on the table. The key insight comes from thinking about fourth down decisions in terms of **expected value**. When a team faces fourth down, they are choosing between three options:

1. **Punt**: Give the ball to the other team, but push them further from your end zone.
2. **Attempt a field goal**: Try to score 3 points, but risk giving the ball to the opponent at the spot of the kick if you miss.
3. **Go for it**: Try to gain enough yards for a first down, but risk turning the ball over at the current spot if you fail.

Each option has a probability of success and a set of possible outcomes. By calculating the expected points for each option, we can determine which decision maximizes a team's expected scoring. As Romer and subsequent researchers have shown, teams should go for it on fourth down far more often than they do, particularly in short-yardage situations and when they are near midfield.

Several coaches have embraced this analytical approach. Kevin Kelley, a high school coach in Arkansas, famously never punted and always attempted onside kicks, arguing that the expected value calculations supported these aggressive strategies. In the NFL, coaches like Brandon Staley (former head coach of the Los Angeles Chargers) adopted more aggressive fourth down strategies informed by analytics, though the results were sometimes mixed and the approach remained controversial.

## Expected Points

The concept of **expected points (EP)** is fundamental to modern football analytics. The idea is simple but powerful: every game state --- defined by the down, distance to go, and field position --- has an associated expected point value that represents the average number of points the team with possession will score on that drive (net of any points the opponent might score).

For example, having a first down at the opponent's 10-yard line is a very favorable situation, and the expected points value will be high (close to 7, since a touchdown is very likely). Conversely, having a third down and 15 yards to go at your own 5-yard line is a poor situation, and the expected points value will be low or even negative (since the opposing team may score after a punt or turnover).

Expected points models are built by analyzing millions of historical plays. For each game state, the model calculates the average net points scored from that state forward. The `nfl_data_py` module provides pre-computed expected points values for every play in its play-by-play data, calculated using the model developed by the `nflfastR` project.

A closely related concept is **expected points added (EPA)**, which measures the value of a single play by taking the difference in expected points before and after the play. If a team starts a play with an EP of 2.5 and ends the play with an EP of 4.0, that play had an EPA of 1.5. EPA has become one of the most important metrics in football analytics because it captures the value of every play in a context-aware manner. A 5-yard gain on third and 4 (which earns a first down) is much more valuable than a 5-yard gain on third and 10 (which does not), and EPA reflects this difference.

## Play-by-Play Data

We will use the `nfl_data_py` module to access NFL play-by-play data. This data set contains detailed information on every play from every NFL game, including the game state (down, distance, field position), the play result (yards gained, touchdowns, turnovers), and pre-computed analytics metrics (expected points, win probability, and more).

We begin by loading play-by-play data from the 2015 through 2025 NFL seasons. This gives us eleven seasons of data, which is sufficient for building robust statistical models.

In [ ]:
df = nfl.import_pbp_data(list(range(2015, 2026)))

Next, we filter the data to extract field goal attempts from the regular season. We compute the field goal distance by adding 17 yards to the `yardline_100` value (the line of scrimmage). The extra 17 yards accounts for the 10-yard end zone and the approximately 7 yards the ball is snapped back to the holder.

The `.assign()` method creates new columns using **lambda functions** --- small, anonymous functions defined inline with the `lambda` keyword. Each `lambda x:` receives the full DataFrame as `x` and returns a new Series. For example, `lambda x: x['yardline_100'] + 17` adds 17 to every value in the `yardline_100` column.

In [ ]:
field_goals = (
    df[
        (df['play_type'] == 'field_goal') &
        (df['season_type'] == 'REG')
    ]
    [['game_date', 'down', 'ydstogo', 'yardline_100', 'posteam',
      'defteam', 'field_goal_result', 'field_goal_attempt',
      'game_seconds_remaining', 'ep', 'fg_prob', 'game_id']]
    .assign(
        success=lambda x: (x['field_goal_result'] == 'made').astype(int),
        fg_distance=lambda x: x['yardline_100'] + 17
    )
    .dropna()
)

We also create a DataFrame of first-and-10 situations, which we will use later to estimate expected points at various field positions.

In [ ]:
first_downs = df[(df['down'] == 1) & (df['ydstogo'] == 10) & (df['ep'] != 0)]

## Field Goal Success vs. Distance

One of the most natural questions in football analytics is: how does the probability of making a field goal depend on the distance of the attempt? We can visualize this relationship and build statistical models to estimate the probability of success.

We begin with a density plot comparing the distribution of field goal distances for made versus missed attempts.

In [ ]:
#| label: fig-fg-density
#| fig-cap: "The density estimates for field goal distance for made versus missed field goals."

fig, ax = plt.subplots(figsize=(10, 6))
sns.kdeplot(data=field_goals, x='fg_distance', hue='success',
            fill=True, alpha=0.5, palette='Set1', ax=ax)
ax.set_xlabel('Distance (yards)')
ax.set_title('Made vs Missed NFL Field Goal Attempts, 2015-2025')
plt.tight_layout()
plt.show()

As the figure shows, made field goals tend to be shorter than missed ones, which is consistent with common sense: longer kicks are harder to make.

### Logistic Regression {.unnumbered}

We can model the probability of making a field goal as a function of distance using **logistic regression**. Logistic regression is appropriate here because the outcome variable (`success`) is binary (0 or 1).

In [ ]:
fg_glm = smf.logit('success ~ fg_distance', data=field_goals).fit()
print(fg_glm.summary())

The negative coefficient on `fg_distance` confirms what we see in the data: as the distance increases, the probability of making the field goal decreases.

We can visualize the fitted logistic regression curve alongside the raw data.

In [ ]:
#| label: fig-fg-distance
#| fig-cap: "The estimated probability of making a field goal as a function of distance in the National Football League."

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(field_goals['fg_distance'], field_goals['success'],
           alpha=0.1, s=10)

x_range = np.linspace(field_goals['fg_distance'].min(),
                      field_goals['fg_distance'].max(), 200)
pred_df = pd.DataFrame({'fg_distance': x_range})
y_pred = fg_glm.predict(pred_df)

ax.plot(x_range, y_pred, color='blue', linewidth=2)
ax.set_xlabel('Distance (yards)')
ax.set_ylabel('Probability of Successful Field Goal')
ax.set_title('Probability of a Successful FG Attempt, NFL 2015-2025')
plt.tight_layout()
plt.show()

### Generalized Additive Model {.unnumbered}

While logistic regression assumes a linear relationship between the log-odds of success and distance, we might expect the relationship to be more complex. For example, the probability might drop off more steeply at very long distances. A **generalized additive model (GAM)** allows us to fit a smooth, nonlinear curve without specifying its functional form in advance.

In [ ]:
X = field_goals[['fg_distance']].values
y = field_goals['success'].values

fg_gam = LogisticGAM(s(0)).fit(X, y)
fg_gam.summary()

We can now compare three models: the `nfl_data_py` pre-computed field goal probability (`fg_prob`), our logistic regression model (GLM), and our GAM.

In [ ]:
field_goals = field_goals.assign(
    glm_pred=fg_glm.predict(field_goals),
    gam_pred=fg_gam.predict_proba(field_goals[['fg_distance']].values)
)

In [ ]:
#| label: fig-fg-distance-all-models
#| fig-cap: "The estimated probabilities of making a field goal as a function of distance in the National Football League. The solid line represents the estimate based on the nfl_data_py model, the dashed line is the GLM model fit, and the dotted line is the GAM model fit."

fig, ax = plt.subplots(figsize=(10, 6))

# Sort for clean line plots
plot_df = field_goals.sort_values('fg_distance')

# nfl_data_py model (loess/smooth)
from scipy.ndimage import uniform_filter1d
fg_smooth = plot_df.groupby('fg_distance')['fg_prob'].mean().reset_index()
ax.plot(fg_smooth['fg_distance'], fg_smooth['fg_prob'],
        color='black', linewidth=2, label='nfl_data_py model')

# GLM predictions
glm_line = plot_df.groupby('fg_distance')['glm_pred'].mean().reset_index()
ax.plot(glm_line['fg_distance'], glm_line['glm_pred'],
        color='black', linewidth=2, linestyle='dashed', label='GLM')

# GAM predictions
gam_line = plot_df.groupby('fg_distance')['gam_pred'].mean().reset_index()
ax.plot(gam_line['fg_distance'], gam_line['gam_pred'],
        color='black', linewidth=2, linestyle='dotted', label='GAM')

ax.set_xlabel('Distance (yards)')
ax.set_ylabel('Probability')
ax.legend()
plt.tight_layout()
plt.show()

The three models produce similar results, particularly in the range of 20 to 50 yards where most field goal attempts occur. The GAM is slightly more flexible and can capture subtle nonlinearities that the simple logistic regression cannot.

## Decision Making and Expected Points

Now that we have models for the probability of making a field goal at any distance, we can use these models alongside expected points to evaluate the decision to attempt a field goal versus other options on fourth down.

Consider a team facing fourth down at the opponent's 33-yard line, which corresponds to a 50-yard field goal attempt. What is the probability of making this kick?

In [ ]:
pred_50 = fg_glm.predict(pd.DataFrame({'fg_distance': [50]}))
print(f"GLM P(make | 50 yards): {pred_50.values[0]:.3f}")

pred_50_gam = fg_gam.predict_proba(np.array([[50]]))
print(f"GAM P(make | 50 yards): {pred_50_gam[0]:.3f}")

If the kick is made, the team scores 3 points. If missed, the opposing team takes over at the spot of the kick (the 33-yard line in this case, unless the kick is returned). What are the expected points from that field position?

In [ ]:
ep_50 = field_goals[field_goals['fg_distance'] == 50]['ep'].mean()
print(f"Mean EP at fg_distance == 50: {ep_50:.3f}")

Now, what if the team decided to go for it instead? If they convert the first down, they would have first-and-10 at approximately the 31-yard line:

In [ ]:
ep_31 = first_downs[first_downs['yardline_100'] == 31]['ep'].mean()
print(f"Mean EP at 1st and 10 from the 31: {ep_31:.3f}")

Alternatively, what if the team punted? A punt from the 33-yard line would likely give the opponent the ball around their own 32-yard line (the opponent's `yardline_100` of 68):

In [ ]:
ep_68 = first_downs[first_downs['yardline_100'] == 68]['ep'].mean()
print(f"Mean EP at 1st and 10 from the opponent's 32: {ep_68:.3f}")

By comparing these expected point values and weighting them by the probabilities of each outcome, coaches can make more informed decisions about whether to kick, punt, or go for it. This is the essence of analytics-driven decision making in football: replacing gut instinct with expected value calculations.

## Simulation

Another powerful tool in football analytics is **simulation**. By randomly sampling from historical play-by-play data, we can simulate drives and estimate how often they result in touchdowns, turnovers, or turnovers on downs. This gives us a way to understand the distribution of outcomes, not just the averages.

We will build three Python functions to simulate NFL drives:

1. `sample_play()`: Randomly samples a single play from historical data matching the current game state.
2. `sample_play_needs_td()`: Wraps `sample_play()` with boundary handling for edge cases (e.g., deep in opponent's territory, long yardage situations).
3. `sample_drive()`: Simulates a complete drive by repeatedly calling `sample_play_needs_td()` until the drive ends.

We load play-by-play data from the 2023 and 2024 seasons for our simulation pool.

In [ ]:
nfl_pbp = nfl.import_pbp_data([2023, 2024, 2025])

We need to compute two additional columns: whether a turnover occurred and whether the offense scored a touchdown.

In [ ]:
nfl_pbp_df = (
    nfl_pbp
    .assign(
        is_fumble=lambda x: x['fumble_recovery_1_team'].notna(),
        turnover=lambda x: (
            x['desc'].str.contains('INTERCEPTED', na=False) |
            (x['is_fumble'] & (x['fumble_recovery_1_team'] != x['posteam']))
        ),
        td_offense=lambda x: (
            x['rush_touchdown'].fillna(0).astype(bool) |
            x['pass_touchdown'].fillna(0).astype(bool)
        )
    )
    .dropna(subset=['yards_gained'])
)

Now we define our three simulation functions.

In [ ]:
def sample_play(down, yards_to_go, yard_line, pbp_data):
    """Sample a single play from historical data matching the game state."""
    eligible = pbp_data[
        (pbp_data['play_type'].isin(['pass', 'run'])) &
        (pbp_data['down'] == down) &
        (pbp_data['ydstogo'] == yards_to_go) &
        (pbp_data['yardline_100'] == yard_line)
    ]
    # If no exact match, relax yardline constraint (+/- 3 yards)
    if len(eligible) == 0:
        eligible = pbp_data[
            (pbp_data['play_type'].isin(['pass', 'run'])) &
            (pbp_data['down'] == down) &
            (pbp_data['ydstogo'] == yards_to_go) &
            (pbp_data['yardline_100'].between(yard_line - 3, yard_line + 3))
        ]
    # If still no match, relax both yardline and distance
    if len(eligible) == 0:
        eligible = pbp_data[
            (pbp_data['play_type'].isin(['pass', 'run'])) &
            (pbp_data['down'] == down) &
            (pbp_data['yardline_100'].between(yard_line - 5, yard_line + 5))
        ]
    return eligible.sample(n=1)


def sample_play_needs_td(down, yards_to_go, yard_line, pbp_data):
    """Sample a play with boundary handling for edge cases."""
    down_original = down
    if yard_line >= 95:
        yard_line = 95
    if down == 4 and yards_to_go >= 3:
        down = 3
    if yards_to_go >= 20:
        yards_to_go = 20

    sim_play = sample_play(down, yards_to_go, yard_line, pbp_data)

    yards_gained = sim_play['yards_gained'].values[0]
    new_yard_line = yard_line - yards_gained
    new_down = 1 if yards_gained >= yards_to_go else down_original + 1

    if yards_gained >= yards_to_go and new_yard_line >= 10:
        new_distance = 10
    elif yards_gained >= yards_to_go and new_yard_line < 10:
        new_distance = new_yard_line
    else:
        new_distance = yards_to_go - yards_gained

    if new_distance <= 0:
        new_distance = 1
    if new_yard_line < 0:
        new_yard_line = 1

    turnover = sim_play['turnover'].values[0]
    td_offense = sim_play['td_offense'].values[0]

    return pd.DataFrame({
        'down_original': [down_original],
        'yards_to_go': [yards_to_go],
        'yard_line': [yard_line],
        'yards_gained': [yards_gained],
        'new_yard_line': [yard_line - yards_gained],
        'new_down': [new_down],
        'new_distance': [new_distance],
        'turnover': [turnover],
        'td_offense': [td_offense],
        'end_drive': [new_down > 4 or turnover or td_offense],
        'play': [sim_play['desc'].values[0]]
    })


def sample_drive(yard_line, pbp_data):
    """Simulate a complete drive from the given yard line."""
    drive_store = []
    new_down = 1
    new_distance = 10
    new_yard_line = yard_line
    end_of_drive = False
    play_num = 1

    while not end_of_drive:
        run_play = sample_play_needs_td(
            new_down, new_distance, new_yard_line, pbp_data
        )
        run_play['play_num'] = play_num
        drive_store.append(run_play)
        new_down = int(run_play['new_down'].values[0])
        new_distance = int(run_play['new_distance'].values[0])
        new_yard_line = int(run_play['new_yard_line'].values[0])
        end_of_drive = bool(run_play['end_drive'].values[0])
        play_num += 1

    return pd.concat(drive_store, ignore_index=True)

Now we simulate 100 drives starting from the offense's own 25-yard line (i.e., 75 yards from the end zone).

In [ ]:
np.random.seed(2024)
n_sim = 100
yard_line_start = 75
results = []

for i in range(n_sim):
    if (i + 1) % 20 == 0:
        print(f"Drive: {i + 1}")
    tmp_result = sample_drive(yard_line_start, nfl_pbp_df)
    tmp_result['sim'] = i + 1
    results.append(tmp_result)

results = pd.concat(results, ignore_index=True)

Let us examine the outcomes of our simulated drives. What percentage ended in a touchdown, a turnover, or a turnover on downs?

In [ ]:
end_plays = results[results['end_drive'] == True]
n = len(end_plays)
print(f"Turnover %:       {end_plays['turnover'].sum() / n:.3f}")
print(f"TD %:             {end_plays['td_offense'].sum() / n:.3f}")
print(f"Turnover on downs: {1 - end_plays['turnover'].sum()/n - end_plays['td_offense'].sum()/n:.3f}")

How many plays does the average drive last?

In [ ]:
print(f"Mean plays per drive: {end_plays['play_num'].mean():.1f}")

In [ ]:
#| label: fig-num-plays-hist
#| fig-cap: "Histogram showing the distribution for number of plays per drive in the simulated data set."

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(end_plays['play_num'], bins=range(1, end_plays['play_num'].max() + 2),
        edgecolor='black', alpha=0.75)
ax.set_xlabel('Number of Plays on Drive')
ax.set_ylabel('Count')
ax.set_title('Mean plays per NFL drive, without kicking, simulated 100 times')
plt.tight_layout()
plt.show()

We can also break down the mean plays per drive by whether or not the drive ended in a touchdown.

In [ ]:
print(end_plays.groupby('td_offense')['play_num'].mean())

In [ ]:
#| label: fig-num-plays-by-td-hist
#| fig-cap: "Histogram showing the distribution for number of plays per drive in the simulated data set. The bars are colored by whether or not the drive ended in a touchdown."

fig, ax = plt.subplots(figsize=(10, 6))
td_plays = end_plays[end_plays['td_offense'] == True]
no_td_plays = end_plays[end_plays['td_offense'] == False]

bins = range(1, end_plays['play_num'].max() + 2)
ax.hist([no_td_plays['play_num'], td_plays['play_num']],
        bins=bins, stacked=True, edgecolor='black', alpha=0.75,
        label=['No TD', 'TD'], color=sns.color_palette('Set1', 2))
ax.set_xlabel('Number of Plays on Drive')
ax.set_ylabel('Count')
ax.set_title('Mean plays per NFL drive, without kicking, simulated 100 times')
ax.legend(title='TD?')
plt.tight_layout()
plt.show()

In [ ]:
#| label: fig-num-plays-facet-hist
#| fig-cap: "Histograms showing the distribution for number of plays per drive in the simulated data set faceted by whether or not the drive ended in a touchdown."

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, (td_val, title) in zip(axes, [(False, 'No Touchdown'), (True, 'Touchdown')]):
    data = end_plays[end_plays['td_offense'] == td_val]
    ax.hist(data['play_num'], bins=range(1, end_plays['play_num'].max() + 2),
            edgecolor='black', alpha=0.75)
    ax.set_xlabel('Number of Plays on Drive')
    ax.set_ylabel('Count')
    ax.set_title(title)

plt.suptitle('Mean plays per NFL drive, without kicking, simulated 100 times')
plt.tight_layout()
plt.show()

The simulation reveals interesting patterns. Drives that end in touchdowns tend to require more plays (the offense needs to march down the field), while drives that end in turnovers or turnovers on downs tend to be shorter. The distribution of drive lengths is right-skewed, with most drives lasting between 3 and 10 plays but some extending well beyond that.

___

::: {.callout-note icon=false}
## Research Block: Using Analytics for Evaluating Defensive Players

Evaluating defensive players using analytics remains one of the most challenging problems in football. While offensive players have clear, countable statistics --- yards, touchdowns, receptions, passer rating --- defensive contributions are often more subtle and harder to quantify.

Traditional defensive statistics like tackles, sacks, and interceptions capture only a fraction of a defender's value. A cornerback who is so good that quarterbacks never throw in his direction will accumulate very few counting stats, yet his impact on the game may be enormous. Similarly, a linebacker who consistently funnels runners toward teammates may not rack up tackles himself but makes the entire defense more effective.

Modern analytics has attempted to address these limitations through several approaches. **Expected Points Added (EPA)** can be calculated for defensive plays, measuring how much a defender's actions reduced the offense's expected scoring. **Defense-adjusted Value Over Average (DVOA)**, developed by Football Outsiders, compares a team's defensive performance to the league average on a play-by-play basis, adjusting for the strength of the opposing offense. **Pro Football Focus (PFF)** employs a team of analysts who grade every player on every play, providing a comprehensive (though subjective) evaluation of defensive performance.

Player tracking data, available since 2016 through the NFL's Next Gen Stats system, has opened new avenues for defensive evaluation. Metrics like **closing speed** (how quickly a defender can reach the ball carrier), **separation allowed** (in coverage), and **pressure rate** (for pass rushers) provide more granular measurements of defensive skill. However, the challenge of separating individual contributions from team scheme effects remains significant.

The difficulty of evaluating defensive players has real consequences for how teams allocate their salary cap dollars. Research has shown that teams may systematically overvalue certain defensive positions (like edge rushers who accumulate sacks) while undervaluing others (like safeties who contribute to coverage). As tracking data and analytical methods continue to improve, we may see shifts in how teams invest in their defenses.
:::

___

## Win Probabilities

Another powerful application of football analytics is the **win probability model**. At every point in a game, we can estimate the probability that each team will win based on the current score, time remaining, field position, and other game state variables. The `nfl_data_py` play-by-play data includes pre-computed win probability estimates for every play, which we can use to visualize how the probability of winning evolves over the course of a game.

### Kansas City vs. Houston, 2019 Playoffs {.unnumbered}

One of the most dramatic games in recent NFL history was the AFC Divisional Round playoff game between the Kansas City Chiefs and the Houston Texans on January 12, 2020. The Texans jumped out to a stunning 24--0 lead in the second quarter, only to watch the Chiefs mount an extraordinary comeback and win 51--31. This game is a perfect case study for win probability visualization.

In [ ]:
nfl_2019 = nfl.import_pbp_data([2019])
kc_hou = nfl_2019[
    (nfl_2019['season_type'] == 'POST') &
    (nfl_2019['home_team'] == 'KC') &
    (nfl_2019['away_team'] == 'HOU')
][['game_seconds_remaining', 'away_wp', 'away_wp_post',
   'home_wp', 'home_wp_post', 'wp', 'desc',
   'home_team', 'away_team']].copy()

In [ ]:
#| label: fig-kc-wp
#| fig-cap: "Kansas City Chiefs win probability versus the Houston Texans from the 2019-20 season playoffs."

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(3600 - kc_hou['game_seconds_remaining'], kc_hou['home_wp'],
        color='black', linewidth=0.5)
ax.scatter(3600 - kc_hou['game_seconds_remaining'], kc_hou['home_wp'],
           s=5, color='black')
ax.set_xlabel('Game Time (seconds)')
ax.set_ylabel('Kansas City Win Probability')
ax.set_title('KC Win Probability, Chiefs vs. Texans, Jan 12 2020')
plt.tight_layout()
plt.show()

The win probability chart clearly shows the Texans' early dominance (KC's win probability drops below 5%) followed by the Chiefs' incredible comeback. We can make this plot more informative by coloring the points according to which team is favored at each moment, using the teams' official colors.

In [ ]:
#| label: fig-kc-wp-tc
#| fig-cap: "Kansas City Chiefs win probability versus the Houston Texans from the 2019-20 season playoffs. The point colors above and below 0.5 are colored according to Kansas City's and Houston's primary team color."

teams = nfl.import_team_desc()
kc_color = teams[teams['team_abbr'] == 'KC']['team_color'].values[0]
hou_color = teams[teams['team_abbr'] == 'HOU']['team_color'].values[0]

g1 = kc_hou.dropna(subset=['home_wp']).copy()
g1['color'] = np.where(g1['home_wp'] >= 0.5, kc_color, hou_color)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(3600 - g1['game_seconds_remaining'], g1['home_wp'],
        color='darkgrey', linewidth=0.5, linestyle='--')
ax.scatter(3600 - g1['game_seconds_remaining'], g1['home_wp'],
           c=g1['color'], s=5)
ax.set_ylim(0, 1)
ax.set_xlim(0, 3600)
ax.set_xticks([0, 900, 1800, 2700, 3600])
ax.set_xticklabels(['Q1', 'Q2', 'Q3', 'Q4', 'FT'])
ax.set_xlabel('Time in Game')
ax.set_ylabel('Kansas City Win Probability')
ax.set_title('KC Win Probability, Chiefs vs. Texans, Jan 12 2020')
plt.tight_layout()
plt.show()

### Seattle at Los Angeles, Week 18 2024 {.unnumbered}

For a more recent example, we can examine the Week 18 game between the Seattle Seahawks and the Los Angeles Rams on January 5, 2025. The Seahawks won 30--25 in a closely contested game.

In [ ]:
nfl_2024 = nfl.import_pbp_data([2024])
sea_la = nfl_2024[
    nfl_2024['game_id'] == '2024_18_SEA_LA'
][['game_seconds_remaining', 'away_wp', 'away_wp_post',
   'home_wp', 'home_wp_post', 'wp', 'desc',
   'home_team', 'away_team']].copy()

In [ ]:
#| label: fig-sea-la
#| fig-cap: "Los Angeles Rams win probability versus the Seattle Seahawks from Week 18 of the 2024 season."

sea_color = teams[teams['team_abbr'] == 'SEA']['team_color'].values[0]
la_color = teams[teams['team_abbr'] == 'LA']['team_color'].values[0]

g2 = sea_la.dropna(subset=['home_wp']).copy()
g2['color'] = np.where(g2['home_wp'] >= 0.5, la_color, sea_color)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(3600 - g2['game_seconds_remaining'], g2['home_wp'],
        color='darkgrey', linewidth=0.5, linestyle='--')
ax.scatter(3600 - g2['game_seconds_remaining'], g2['home_wp'],
           c=g2['color'], s=5)
ax.set_ylim(0, 1)
ax.set_xlim(0, 3600)
ax.set_xticks([0, 900, 1800, 2700, 3600])
ax.set_xticklabels(['Q1', 'Q2', 'Q3', 'Q4', 'FT'])
ax.set_xlabel('Time in Game')
ax.set_ylabel('LA Rams Win Probability')
ax.set_title('Win Probability: SEA @ LA, Week 18 2024')
plt.tight_layout()
plt.show()

We can also visualize both teams' win probabilities simultaneously. Since the two probabilities always sum to 1, this creates a mirror image effect that makes it easy to see how the game swung between the two teams.

In [ ]:
#| label: fig-sea-la-both
#| fig-cap: "Los Angeles Rams and Seattle Seahawks win probabilities from Week 18 of the 2024 season."

g3 = sea_la.dropna(subset=['home_wp']).copy()

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(3600 - g3['game_seconds_remaining'], g3['home_wp'],
        color=la_color, linewidth=1.5, label='LA Rams')
ax.plot(3600 - g3['game_seconds_remaining'], g3['away_wp'],
        color=sea_color, linewidth=1.5, label='SEA Seahawks')
ax.set_ylim(0, 1)
ax.set_xlim(0, 3600)
ax.set_xticks([0, 900, 1800, 2700, 3600])
ax.set_xticklabels(['Q1', 'Q2', 'Q3', 'Q4', 'FT'])
ax.set_xlabel('Time in Game')
ax.set_ylabel('Win Probability')
ax.set_title('Win Probability: SEA @ LA, Week 18 2024')
ax.legend()
plt.tight_layout()
plt.show()

## Conclusion

In this chapter, we explored several key areas of football analytics using Python. We began by examining the fundamental concepts of **expected points** and **expected points added**, which provide a framework for evaluating the value of every play in context. We then built statistical models --- both logistic regression and generalized additive models --- to estimate the probability of making a field goal at any distance, and used these models alongside expected points to evaluate coaching decisions on fourth down.

Our simulation exercise demonstrated how we can use historical play-by-play data to simulate NFL drives and estimate the distribution of outcomes. This approach can be extended to answer more complex strategic questions, such as the value of always going for it on fourth down or the expected impact of different play-calling strategies.

Finally, we explored **win probability models** through two case studies: the extraordinary 2019 playoff comeback by the Kansas City Chiefs and a tightly contested 2024 regular season game between the Seattle Seahawks and Los Angeles Rams. These visualizations capture the drama of football in a single chart and illustrate how analytics can quantify the ebb and flow of a game.

Football analytics continues to evolve rapidly. The availability of player tracking data, advances in machine learning, and the growing acceptance of data-driven decision-making by coaching staffs are all driving the field forward. As you work through the exercises at the end of this chapter, you will have the opportunity to explore many of these topics in greater depth.

___

::: {.callout-note icon=false}
## Case Study: Overtime Strategy in the NFL

The NFL's overtime rules have been a source of controversy and analytical debate for years. Prior to 2022, the overtime format worked as follows: a coin toss determined which team received the ball first, and if the receiving team scored a touchdown on its opening drive, the game was over. If instead the receiving team kicked a field goal, the other team had one possession to respond. If neither team scored, the game continued in a sudden-death format.

This format was widely criticized because the coin toss had an outsized influence on the outcome. Analysis showed that the team winning the coin toss won approximately 52--55% of overtime games, a meaningful advantage in a sport with thin margins.

In 2022, the NFL modified the playoff overtime rules so that both teams are guaranteed at least one possession, regardless of what happens on the first drive. This change was prompted in part by the dramatic 2021 AFC Championship game between the Kansas City Chiefs and Buffalo Bills, in which the Chiefs won the coin toss and scored a touchdown on the opening drive, ending the game without Josh Allen ever touching the ball in overtime.

The regular season overtime rules remained largely unchanged until 2024, when the league extended the "both teams get a possession" rule to regular season games as well.

From an analytics perspective, the overtime rules raise several interesting questions:

1. **Under the old rules, should the team that wins the coin toss always elect to receive?** While the conventional wisdom said yes, some analysts argued that in certain situations --- particularly for teams with strong defenses and weaker offenses --- deferring could be the better strategy.

2. **How much did the 2022 rule change affect competitive balance?** By guaranteeing both teams a possession in the playoffs, the coin toss advantage was reduced. But did it create new strategic considerations? For example, if your team scores a touchdown first, the opposing team now knows it *must* score a touchdown or lose, which changes their play-calling calculus.

3. **What is the optimal strategy on the opening possession of overtime?** Should teams play aggressively to score a touchdown, or should they manage risk and settle for a field goal, knowing the other team will also get a chance? The answer depends on the specific rule set, the strength of both teams' offenses and defenses, and the game situation.
:::

___

## Exercises

1. Filter the play-by-play data for punt plays (`play_type == 'punt'`) from 2015 to 2025. Create a density plot of punt distance. Then fit a logistic regression model to predict the probability of a touchback as a function of `kick_distance`.

In [ ]:
#| eval: false
#| code-fold: true
#| code-summary: "Show solution"

punts = df[df['play_type'] == 'punt'].dropna(subset=['kick_distance', 'touchback'])
punts = punts.assign(tb=lambda x: x['touchback'].astype(int))

fig, ax = plt.subplots(figsize=(10, 6))
sns.kdeplot(data=punts, x='kick_distance', fill=True, alpha=0.5, ax=ax)
ax.set_xlabel('Kick Distance (yards)')
ax.set_title('Distribution of NFL Punt Distances, 2015-2025')
plt.tight_layout()
plt.show()

punt_glm = smf.logit('tb ~ kick_distance', data=punts).fit()
print(punt_glm.summary())

2. Calculate the mean EPA per play for each team in the 2025 season. Create a horizontal bar chart showing the top 10 teams by offensive EPA per play.

In [ ]:
#| eval: false
#| code-fold: true
#| code-summary: "Show solution"

nfl_2025 = nfl.import_pbp_data([2025])
plays_2025 = nfl_2025[
    (nfl_2025['play_type'].isin(['pass', 'run'])) &
    (nfl_2025['season_type'] == 'REG')
]
epa_team = (plays_2025.groupby('posteam')['epa']
            .mean()
            .sort_values(ascending=True)
            .tail(10))

fig, ax = plt.subplots(figsize=(10, 6))
epa_team.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('EPA per Play')
ax.set_ylabel('Team')
ax.set_title('Top 10 NFL Teams by Offensive EPA per Play, 2025')
plt.tight_layout()
plt.show()

3. Modify the `sample_drive()` function to include a field goal attempt option. On fourth down, if the team is within field goal range (inside the opponent's 35-yard line, i.e., `yard_line <= 35`), attempt a field goal using the GLM model to determine the probability of success. If the field goal is made, record it as a successful scoring drive. Run 100 simulations from the 25-yard line and compare the results to the version without field goals.

In [ ]:
#| eval: false
#| code-fold: true
#| code-summary: "Show solution"

def sample_drive_with_fg(yard_line, pbp_data, fg_model):
    """Simulate a drive with field goal attempts on 4th down."""
    drive_store = []
    new_down = 1
    new_distance = 10
    new_yard_line = yard_line
    end_of_drive = False
    play_num = 1
    fg_made = False

    while not end_of_drive:
        # Check for FG attempt on 4th down
        if new_down == 4 and new_yard_line <= 35:
            fg_dist = new_yard_line + 17
            fg_prob = fg_model.predict(
                pd.DataFrame({'fg_distance': [fg_dist]})
            ).values[0]
            if np.random.random() < fg_prob:
                fg_made = True
            end_of_drive = True
            drive_store.append(pd.DataFrame({
                'play_num': [play_num], 'fg_made': [fg_made],
                'td_offense': [False], 'turnover': [False],
                'end_drive': [True]
            }))
            break

        run_play = sample_play_needs_td(
            new_down, new_distance, new_yard_line, pbp_data
        )
        run_play['play_num'] = play_num
        drive_store.append(run_play)
        new_down = int(run_play['new_down'].values[0])
        new_distance = int(run_play['new_distance'].values[0])
        new_yard_line = int(run_play['new_yard_line'].values[0])
        end_of_drive = bool(run_play['end_drive'].values[0])
        play_num += 1

    return pd.concat(drive_store, ignore_index=True)

4. Create a win probability plot for a game of your choosing from the 2025 season. Use `nfl.import_schedules([2025])` to find game IDs, then filter the play-by-play data for that game and create a team-colored win probability chart.

In [ ]:
#| eval: false
#| code-fold: true
#| code-summary: "Show solution"

schedule = nfl.import_schedules([2025])
# Example: find a specific game
# game = schedule[(schedule['home_team'] == 'DET') & (schedule['week'] == 1)]
# game_id = game['game_id'].values[0]
# Then filter nfl_2025 by game_id and plot as shown above

5. Compare field goal success rates between games played in domed stadiums versus outdoor stadiums. The `roof` column in the play-by-play data indicates the stadium type. Create a bar chart comparing success rates and perform a chi-squared test to determine if the difference is statistically significant.

In [ ]:
#| eval: false
#| code-fold: true
#| code-summary: "Show solution"

fg_roof = (
    df[(df['play_type'] == 'field_goal') & (df['season_type'] == 'REG')]
    .assign(success=lambda x: (x['field_goal_result'] == 'made').astype(int))
    .dropna(subset=['roof'])
)
fg_roof['venue_type'] = np.where(
    fg_roof['roof'].isin(['dome', 'closed']), 'Indoor', 'Outdoor'
)

success_rates = fg_roof.groupby('venue_type')['success'].mean()
print(success_rates)

fig, ax = plt.subplots(figsize=(8, 5))
success_rates.plot(kind='bar', ax=ax, color=['steelblue', 'coral'])
ax.set_ylabel('Success Rate')
ax.set_title('FG Success Rate: Indoor vs Outdoor')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

# Chi-squared test
ct = pd.crosstab(fg_roof['venue_type'], fg_roof['success'])
chi2, p, dof, expected = stats.chi2_contingency(ct)
print(f"Chi-squared: {chi2:.3f}, p-value: {p:.4f}")

6. Simulate 1,000 drives starting from the 50-yard line and compute the probability of scoring a touchdown. Compare this to the TD probability from drives starting at the 25-yard line (75 yards from the end zone). How much does starting field position matter?

In [ ]:
#| eval: false
#| code-fold: true
#| code-summary: "Show solution"

np.random.seed(42)
td_counts = {'50-yard line': 0, '25-yard line': 0}

for i in range(1000):
    drive = sample_drive(50, nfl_pbp_df)
    end = drive[drive['end_drive'] == True]
    if end['td_offense'].values[0]:
        td_counts['50-yard line'] += 1

for i in range(1000):
    drive = sample_drive(75, nfl_pbp_df)
    end = drive[drive['end_drive'] == True]
    if end['td_offense'].values[0]:
        td_counts['25-yard line'] += 1

for pos, count in td_counts.items():
    print(f"TD probability from {pos}: {count / 1000:.3f}")

7. Fit a GAM to model expected points (`ep`) as a function of field position (`yardline_100`) for first-and-10 situations. Plot the fitted curve. Where on the field does the expected points value cross zero?

In [ ]:
#| eval: false
#| code-fold: true
#| code-summary: "Show solution"

from pygam import LinearGAM, s as gam_s

fd = first_downs.dropna(subset=['ep', 'yardline_100']).copy()
X_ep = fd[['yardline_100']].values
y_ep = fd['ep'].values

ep_gam = LinearGAM(gam_s(0)).fit(X_ep, y_ep)

x_grid = np.linspace(1, 99, 200).reshape(-1, 1)
y_grid = ep_gam.predict(x_grid)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x_grid, y_grid, color='blue', linewidth=2)
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Yards from End Zone (yardline_100)')
ax.set_ylabel('Expected Points')
ax.set_title('Expected Points by Field Position (1st and 10)')
plt.tight_layout()
plt.show()

# Find where EP crosses zero
zero_idx = np.argmin(np.abs(y_grid))
print(f"EP crosses zero at approximately: {x_grid[zero_idx][0]:.0f} yards from end zone")